In [1]:
import numpy as np
import cv2
import glob

In [2]:
PATTERN_SIZE = (9,6)
left_imgs = list(sorted(glob.glob('./calibration_images/*left.jpeg')))
right_imgs = list(sorted(glob.glob('./calibration_images/*right.jpeg')))
assert len(left_imgs)==len(right_imgs)

In [3]:
criteria = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
left_pts, right_pts = [],[]
img_size = None
counter = 0
for left_img_path, right_img_path in zip(left_imgs, right_imgs):
    left_img = cv2.imread(left_img_path, cv2.IMREAD_GRAYSCALE)
    right_img = cv2.imread(right_img_path, cv2.IMREAD_GRAYSCALE)
    if img_size is None:
        img_size = (left_img.shape[1], left_img.shape[0])
    res_left, corners_left = cv2.findChessboardCorners(left_img, PATTERN_SIZE)
    res_right, corners_right = cv2.findChessboardCorners(right_img, PATTERN_SIZE)
    if res_left and res_right:
        corners_left = cv2.cornerSubPix(left_img, corners_left, (10,10), (-1,-1), criteria)
        corners_right = cv2.cornerSubPix(right_img, corners_right, (10,10), (-1,-1), criteria)
        
        left_pts.append(corners_left)
        right_pts.append(corners_right)
        print(counter)
        counter += 1

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29


In [4]:
pattern_points = np.zeros((np.prod(PATTERN_SIZE), 3), np.float32) 
pattern_points[:,:2] = np.indices(PATTERN_SIZE).T.reshape(-1, 2)
pattern_points = [pattern_points]*len(left_pts)
pattern_points

[array([[0., 0., 0.],
        [1., 0., 0.],
        [2., 0., 0.],
        [3., 0., 0.],
        [4., 0., 0.],
        [5., 0., 0.],
        [6., 0., 0.],
        [7., 0., 0.],
        [8., 0., 0.],
        [0., 1., 0.],
        [1., 1., 0.],
        [2., 1., 0.],
        [3., 1., 0.],
        [4., 1., 0.],
        [5., 1., 0.],
        [6., 1., 0.],
        [7., 1., 0.],
        [8., 1., 0.],
        [0., 2., 0.],
        [1., 2., 0.],
        [2., 2., 0.],
        [3., 2., 0.],
        [4., 2., 0.],
        [5., 2., 0.],
        [6., 2., 0.],
        [7., 2., 0.],
        [8., 2., 0.],
        [0., 3., 0.],
        [1., 3., 0.],
        [2., 3., 0.],
        [3., 3., 0.],
        [4., 3., 0.],
        [5., 3., 0.],
        [6., 3., 0.],
        [7., 3., 0.],
        [8., 3., 0.],
        [0., 4., 0.],
        [1., 4., 0.],
        [2., 4., 0.],
        [3., 4., 0.],
        [4., 4., 0.],
        [5., 4., 0.],
        [6., 4., 0.],
        [7., 4., 0.],
        [8., 4., 0.],
        [0

In [5]:
err, Kl, Dl, Kr, Dr, R, T, E, F = cv2.stereoCalibrate(
    pattern_points, left_pts, right_pts, None, None, None, None, img_size, flags=0
)

In [6]:
img_size

(1440, 1080)

In [7]:
print('Left camera:')
print(Kl)
print('Left camera distortion:')
print(Dl)
print('Right camera:')
print(Kr)
print('Right camera distortion:')
print(Dr)
print('Rotation matrix:')
print(R)
print('Translation:')
print(T)

Left camera:
[[1.81171190e+03 0.00000000e+00 7.87697208e+02]
 [0.00000000e+00 1.77710712e+03 5.51311153e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Left camera distortion:
[[-7.84004553e-01  2.70825704e+00 -3.75987077e-02 -4.03248781e-03
  -7.33709650e+00]]
Right camera:
[[1.82862822e+03 0.00000000e+00 6.19754112e+02]
 [0.00000000e+00 1.87267770e+03 5.27774084e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Right camera distortion:
[[-0.69069329  0.12825757 -0.03935826  0.01884002  4.24934518]]
Rotation matrix:
[[ 0.99940137  0.02467074 -0.02425389]
 [-0.02406149  0.99939537  0.02509854]
 [ 0.02485842 -0.02449993  0.99939072]]
Translation:
[[ 4.51820292]
 [-0.32701622]
 [ 2.32158652]]


In [8]:
np.save('./stereo.npy', {'Kl': Kl, 'Dl':Dl, 'Kr':Kr, 'Dr': Dr, 'R':R, 'T':T, 'E':E, 'F':F, 'img_size':img_size, 'left_pts': left_pts, 'right_pts':right_pts})